# Analyse des hyperparamètres — RBF & MLP

Ce notebook explore l'impact des hyperparamètres sur les performances des modèles RBF et MLP.

**Hyperparamètres étudiés :**
- **RBF** : `n_centers` (nombre de centres), `lr` (taux d'apprentissage), `epochs`
- **MLP** : `lr`, `epochs`, architecture des couches cachées

**Objectif** : trouver la configuration optimale pour maximiser l'accuracy sur le jeu de test.

In [ ]:
import os, sys, subprocess, time
import numpy as np
import matplotlib.pyplot as plt

IN_COLAB = os.path.exists('/content')

if IN_COLAB:
    BASE_DIR = '/content/VisionAI'
    if not os.path.exists(BASE_DIR):
        subprocess.run(['git', 'clone', '-b', 'RBF_thinina',
                        'https://github.com/SINCER-Ali/VisionAI.git', BASE_DIR], check=True)

    # Installer Rust si absent (requis par maturin)
    cargo = os.path.expanduser('~/.cargo/bin/cargo')
    if not os.path.exists(cargo):
        print('Installation de Rust...')
        subprocess.run(
            'curl https://sh.rustup.rs -sSf | sh -s -- -y --default-toolchain stable',
            shell=True, check=True
        )
    os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']

    try:
        import vision_ai
        print('vision_ai déjà disponible')
    except ImportError:
        print('Compilation du binding Python...')
        subprocess.run(['pip', 'install', 'maturin', '-q'], check=True)
        subprocess.run(
            [os.path.expanduser('~/.cargo/bin/maturin'), 'develop', '--release'],
            cwd=f'{BASE_DIR}/python_binding', check=True
        )
else:
    BASE_DIR = os.path.dirname(os.path.abspath('.'))

DATASET_DIR = os.path.join(BASE_DIR, 'datasets')
sys.path.insert(0, os.path.join(BASE_DIR, 'python_binding'))

import vision_ai
CLASSES = ['aucun', 'humain', 'animal']
print('vision_ai importé ✓')

## 1. Chargement du dataset

In [ ]:
X_train = np.load(os.path.join(DATASET_DIR, 'X_train.npy'))
y_train = np.load(os.path.join(DATASET_DIR, 'y_train.npy'))
X_test  = np.load(os.path.join(DATASET_DIR, 'X_test.npy'))
y_test  = np.load(os.path.join(DATASET_DIR, 'y_test.npy'))

# Sous-échantillonnage (1 pixel sur 4) pour réduire la dim RBF
STEP = 4
X_train_r = X_train[:, ::STEP]
X_test_r  = X_test[:, ::STEP]
DIM_RBF   = X_train_r.shape[1]
INPUT_SIZE = X_train.shape[1]

inputs_train_r = X_train_r.tolist()
inputs_test_r  = X_test_r.tolist()
inputs_train   = X_train.tolist()
inputs_test    = X_test.tolist()

def one_hot(labels, n=3):
    return [[1.0 if int(l)==i else 0.0 for i in range(n)] for l in labels]

targets_train = one_hot(y_train)

def accuracy_fn(predict_fn, inputs, labels):
    preds = [predict_fn(x) for x in inputs]
    y_pred = [p.index(max(p)) for p in preds]
    return sum(p == int(t) for p, t in zip(y_pred, labels)) / len(labels)

print(f'Train : {X_train.shape}  |  Test : {X_test.shape}')
print(f'Dim RBF (réduite) : {DIM_RBF}  |  Dim MLP (plein) : {INPUT_SIZE}')

## 2. Impact du nombre de centres — RBF

In [ ]:
n_centers_list = [5, 10, 20, 30, 50]
accs_centers   = []
times_centers  = []

for n in n_centers_list:
    t0 = time.time()
    rbf = vision_ai.PyRBF(DIM_RBF, 3, n_centers=n, sigma=1.0)
    rbf.init_centers_random(inputs_train_r)
    rbf.train(inputs_train_r, targets_train, lr=0.01, epochs=80, regression=False)
    elapsed = time.time() - t0
    acc = accuracy_fn(rbf.predict, inputs_test_r, y_test)
    accs_centers.append(acc * 100)
    times_centers.append(elapsed)
    print(f'n_centers={n:3d}  →  acc={acc*100:.1f}%  ({elapsed:.1f}s)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(n_centers_list, accs_centers, 'o-', color='darkorange')
axes[0].set_xlabel('Nombre de centres (n_centers)')
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('RBF — Accuracy vs n_centers')
axes[0].grid(True, alpha=0.3)

axes[1].plot(n_centers_list, times_centers, 's-', color='steelblue')
axes[1].set_xlabel('Nombre de centres (n_centers)')
axes[1].set_ylabel("Temps d'entraînement (s)")
axes[1].set_title("RBF — Temps vs n_centers")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_idx = accs_centers.index(max(accs_centers))
print(f'\nMeilleur n_centers : {n_centers_list[best_idx]} → {accs_centers[best_idx]:.1f}%')

## 3. Impact du learning rate — RBF

In [ ]:
lr_list    = [0.001, 0.005, 0.01, 0.05, 0.1]
accs_lr_rbf = []

for lr in lr_list:
    rbf = vision_ai.PyRBF(DIM_RBF, 3, n_centers=30, sigma=1.0)
    rbf.init_centers_random(inputs_train_r)
    rbf.train(inputs_train_r, targets_train, lr=lr, epochs=80, regression=False)
    acc = accuracy_fn(rbf.predict, inputs_test_r, y_test)
    accs_lr_rbf.append(acc * 100)
    print(f'lr={lr:.3f}  →  acc={acc*100:.1f}%')

plt.figure(figsize=(7, 4))
plt.semilogx(lr_list, accs_lr_rbf, 'o-', color='darkorange')
plt.xlabel('Learning rate (échelle log)')
plt.ylabel('Accuracy (%)')
plt.title('RBF — Accuracy vs Learning Rate')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_idx = accs_lr_rbf.index(max(accs_lr_rbf))
print(f'\nMeilleur lr pour RBF : {lr_list[best_idx]} → {accs_lr_rbf[best_idx]:.1f}%')

## 4. Impact du nombre d'époques — RBF

In [ ]:
epochs_list   = [20, 50, 100, 200, 300]
accs_ep_rbf   = []

for ep in epochs_list:
    rbf = vision_ai.PyRBF(DIM_RBF, 3, n_centers=30, sigma=1.0)
    rbf.init_centers_random(inputs_train_r)
    rbf.train(inputs_train_r, targets_train, lr=0.01, epochs=ep, regression=False)
    acc = accuracy_fn(rbf.predict, inputs_test_r, y_test)
    accs_ep_rbf.append(acc * 100)
    print(f'epochs={ep:4d}  →  acc={acc*100:.1f}%')

plt.figure(figsize=(7, 4))
plt.plot(epochs_list, accs_ep_rbf, 'o-', color='darkorange')
plt.xlabel('Nombre d\'époques')
plt.ylabel('Accuracy (%)')
plt.title('RBF — Accuracy vs Epochs')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Impact du learning rate — MLP

In [ ]:
lr_list_mlp  = [0.0001, 0.0005, 0.001, 0.005, 0.01]
accs_lr_mlp  = []

for lr in lr_list_mlp:
    mlp = vision_ai.PyMLP([INPUT_SIZE, 64, 3])
    mlp.train(inputs_train, targets_train, learning_rate=lr, epochs=15)
    acc = accuracy_fn(mlp.predict, inputs_test, y_test)
    accs_lr_mlp.append(acc * 100)
    print(f'lr={lr:.4f}  →  acc={acc*100:.1f}%')

plt.figure(figsize=(7, 4))
plt.semilogx(lr_list_mlp, accs_lr_mlp, 'o-', color='seagreen')
plt.xlabel('Learning rate (échelle log)')
plt.ylabel('Accuracy (%)')
plt.title('MLP — Accuracy vs Learning Rate')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_idx = accs_lr_mlp.index(max(accs_lr_mlp))
print(f'\nMeilleur lr pour MLP : {lr_list_mlp[best_idx]} → {accs_lr_mlp[best_idx]:.1f}%')

## 6. Impact de l'architecture MLP (couches cachées)

In [ ]:
architectures = [
    [INPUT_SIZE, 32, 3],
    [INPUT_SIZE, 64, 3],
    [INPUT_SIZE, 128, 3],
    [INPUT_SIZE, 64, 32, 3],
    [INPUT_SIZE, 128, 64, 3],
]
arch_labels = ['32', '64', '128', '64-32', '128-64']
accs_arch   = []

for arch, label in zip(architectures, arch_labels):
    mlp = vision_ai.PyMLP(arch)
    mlp.train(inputs_train, targets_train, learning_rate=0.001, epochs=15)
    acc = accuracy_fn(mlp.predict, inputs_test, y_test)
    accs_arch.append(acc * 100)
    print(f'Architecture {label:8s}  →  acc={acc*100:.1f}%')

plt.figure(figsize=(8, 4))
bars = plt.bar(arch_labels, accs_arch, color='seagreen', width=0.5)
plt.ylim(0, 100)
plt.xlabel('Couches cachées')
plt.ylabel('Accuracy (%)')
plt.title('MLP — Accuracy vs Architecture')
for bar, acc in zip(bars, accs_arch):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{acc:.1f}%', ha='center', fontweight='bold', fontsize=9)
plt.tight_layout()
plt.show()

best_idx = accs_arch.index(max(accs_arch))
print(f'\nMeilleure architecture : {arch_labels[best_idx]} → {accs_arch[best_idx]:.1f}%')

## 7. Récapitulatif des meilleurs hyperparamètres

In [ ]:
print('========== RÉCAPITULATIF DES MEILLEURS HYPERPARAMÈTRES ==========')
print()
print('RBF :')
best_nc  = n_centers_list[accs_centers.index(max(accs_centers))]
best_lr_rbf = lr_list[accs_lr_rbf.index(max(accs_lr_rbf))]
best_ep  = epochs_list[accs_ep_rbf.index(max(accs_ep_rbf))]
print(f'  n_centers optimal : {best_nc}')
print(f'  lr optimal        : {best_lr_rbf}')
print(f'  epochs optimal    : {best_ep}')
print()
print('MLP :')
best_lr_mlp  = lr_list_mlp[accs_lr_mlp.index(max(accs_lr_mlp))]
best_arch    = arch_labels[accs_arch.index(max(accs_arch))]
print(f'  lr optimal        : {best_lr_mlp}')
print(f'  architecture      : [{best_arch}]')
print()
print('Ces valeurs servent de configuration de référence pour le notebook de comparaison.')